<a href="https://colab.research.google.com/github/faisu6339-glitch/Deep-Learning/blob/main/LSTM_(Text_Classification).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

Given a sentence, predict whether it is Positive (1) or Negative (0).

### Example Dataset

| Sentence           | Label |
|--------------------|-------|
| I love this movie  | 1     |
| This film is amazing | 1     |
| I hate this movie  | 0     |
| Worst movie ever   | 0     |
| Excellent acting   | 1     |
| Bad story          | 0     |

Here,

Positive = 1

Negative = 0

#Step 1: Import Libraries

In [1]:
import numpy as np
import pandas as pd

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

#Step 2: Create Dataset

In [2]:
data = {
    "Text":[
        "I love this movie",
        "This film is amazing",
        "I hate this movie",
        "Worst movie ever",
        "Excellent acting",
        "Bad story"
    ],

    "Label":[1,1,0,0,1,0]
}

df = pd.DataFrame(data)

print(df)

                   Text  Label
0     I love this movie      1
1  This film is amazing      1
2     I hate this movie      0
3      Worst movie ever      0
4      Excellent acting      1
5             Bad story      0


#Step 3: Separate Input and Output

In [3]:
X=df['Text']
y=df['Label']

print(X)
print(y)

0       I love this movie
1    This film is amazing
2       I hate this movie
3        Worst movie ever
4        Excellent acting
5               Bad story
Name: Text, dtype: object
0    1
1    1
2    0
3    0
4    1
5    0
Name: Label, dtype: int64


#Step 4: Tokenization


In [5]:
tokenizer=Tokenizer()
tokenizer.fit_on_texts(X)
word_index=tokenizer.word_index

print(word_index)

{'this': 1, 'movie': 2, 'i': 3, 'love': 4, 'film': 5, 'is': 6, 'amazing': 7, 'hate': 8, 'worst': 9, 'ever': 10, 'excellent': 11, 'acting': 12, 'bad': 13, 'story': 14}


Notice

Each unique word gets a unique integer.

#Step 5: Convert Sentences into Sequences

In [6]:
from typing import Sequence
Sequences=tokenizer.texts_to_sequences(X)

print(Sequences)

[[3, 4, 1, 2], [1, 5, 6, 7], [3, 8, 1, 2], [9, 2, 10], [11, 12], [13, 14]]


Explanation

Sentence

I love this movie

becomes

[1,4,2,3]

## Step 6: Why Padding?

Look carefully.

Sentence 1

[1,4,2,3]

Length = 4

Sentence 2

[9,3,10]

Length = 3

Sentence 3

[11,12]

Length = 2

All sentences have different lengths.

LSTM expects equal-length input.

So we use Padding.

#Step 7: Apply Padding

In [7]:
X_pad=pad_sequences(Sequences)

print(X_pad)

[[ 3  4  1  2]
 [ 1  5  6  7]
 [ 3  8  1  2]
 [ 0  9  2 10]
 [ 0  0 11 12]
 [ 0  0 13 14]]


Explanation

Zeros are added at the beginning so every sequence has the same length.

#Step 8: Build LSTM Model

In [8]:
model = Sequential([
    Embedding(input_dim=20,
              output_dim=8,
              input_length=4),

    LSTM(16),

    Dense(1, activation="sigmoid")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Let's understand every layer.

### Embedding Layer
`Embedding(
input_dim=20,
output_dim=8,
input_length=4
)`

*   **`input_dim`**: This represents the **vocabulary size**. It's the total number of unique words the model will consider. For instance, if you have 14 unique words in your dataset, adding 1 for out-of-vocabulary words or padding typically makes it 15. We usually choose a value slightly larger than the actual vocabulary size, such as 20, to allow for some flexibility and future additions.

*   **`output_dim`**: This is the **dimension of the dense embedding**. Each word from your vocabulary will be converted into an `output_dim`-dimensional vector. For example, if `output_dim=8`, a word like "movie" might become a vector like `[0.25, 0.71, 0.12, ..., 0.88]`. This is much richer than just its integer ID (e.g., `3`) and helps the model learn semantic relationships between words.

*   **`input_length`**: This specifies the **length of input sequences** after padding. Since LSTM expects equal-length input, all your text sequences are padded or truncated to this length (in this case, 4).

### LSTM Layer
`LSTM(16)`

*   This layer is a Long Short-Term Memory (LSTM) recurrent neural network with **16 memory cells (or units)**. The LSTM processes the word embeddings sequentially, maintaining an internal state (memory) that allows it to capture long-range dependencies in the text. It reads one word at a time, updates its memory, and then passes its hidden state (a feature vector) to the next layer.

### Dense Layer
`Dense(
1,
activation="sigmoid"
)`

*   This is a standard **dense (fully connected) neural network layer**.
    *   The `1` indicates that it has **one neuron**, which is suitable for binary classification tasks (like positive/negative sentiment).
    *   The `activation="sigmoid"` means this neuron will output a value between **0 and 1**, which can be interpreted as the probability of the input text belonging to the positive class (1). If the output is closer to 1, it's more likely positive; if closer to 0, it's more likely negative.

In [9]:
model.compile(
optimizer="adam",
loss="binary_crossentropy",
metrics=["accuracy"]
)

In [13]:
model.fit(
    X_pad,
    y,
    epochs=200,
    verbose=1
)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8333 - loss: 0.6865
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.8333 - loss: 0.6860
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8333 - loss: 0.6854
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.8333 - loss: 0.6848
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8333 - loss: 0.6842
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8333 - loss: 0.6835
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 1.0000 - loss: 0.6828
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.6821
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 1.0000 - loss: 0.6813
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.6805
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 0.6796
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - lo

#: Test the Model

In [14]:
test = ["I love acting"]

seq = tokenizer.texts_to_sequences(test)

pad = pad_sequences(seq, maxlen=4)

prediction = model.predict(pad)

print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
[[0.9767062]]


In [15]:
prediction = model.predict(pad)

print("Probability:", prediction[0][0])

if prediction[0][0] >= 0.5:
    print("Prediction: Positive 😊")
else:
    print("Prediction: Negative 😞")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Probability: 0.9767062
Prediction: Positive 😊
